In [ ]:
#| hide
import llmdojo.dojo, llmdojo.rules, llmdojo.claudedojo, llmdojo.codexdojo

# llmdojo

> Scored tooling katas, habit rules, and pre-baked warm-start sessions for LLM coding agents

LLM coding agents imitate what their context shows far more reliably than what it tells: instructions decay over a session, demonstrations don't. llmdojo turns that observation into tooling, for kernels served by [clikernel](https://github.com/AnswerDotAI/clikernel):

- `llmdojo.rules`: session rules, applied live to every kernel cell through clikernel's inspector hook. Routing rules teach the designed tool for each job, string-safety rules catch quoting mistakes, gating rules block the genuinely dangerous, and the `nodoc` rule tracks per-conversation doc-state so tooling docs actually get read before first use.
- `llmdojo.dojo`: a short scored practice round (the katas) an agent completes at session start, so its first real tool calls follow a demonstrated clean pass rather than being produced cold. Clean rounds mint a completion id that later sessions can present instead of replaying.
- `llmdojo.claudedojo`: capture a clean round from a live Claude Code session, curate it into a deterministic template, and launch new sessions that *resume* it, so every session opens with the worked round already in context.
- `llmdojo.codexdojo`: capture a clean Codex thread into a native Responses-item template, then launch new Codex sessions from the reviewed template.

## Usage

### Installation

```sh
$ pip install llmdojo
```

Activation is two lines of clikernel user config: register the rules in `$XDG_CONFIG_HOME/clikernel/inspectors.py` (`from llmdojo.rules import make_inspector, RuleBlock; inspectors = [make_inspector()]`), and have your startup file print the bootstrap instructions that tell the agent to run the dojo.

### Start a session

`dojo_start()` deals a scored practice round inside a clikernel session. Pass the completion id from a clean round to skip replaying it while that receipt remains valid.

Claude Code and Codex can start with a reviewed round already in their history:

```sh
$ claudedojo
$ codexdojo
```

Arguments after either command pass through to the underlying agent. After compaction, append the worked round to the existing conversation and resume it:

```sh
$ claude -r $(claudedojo -r)
$ codex resume $(codexdojo -r)
```

Template maintainers can capture fresh rounds with `claudedojo --capture` and `codexdojo --capture`, import an existing Codex round with `codexdojo --capture-current`, and compile a reviewed dialog with `--build <dialog.ipynb>`.

### State and templates

Doc-state records which tool documentation the agent has read for a conversation. Closing and resuming restores that record; compaction clears it because the corresponding documentation has left the model's context. A warm-start template seeds both doc-state and its clean-round completion receipt before the agent resumes.

Completion receipts and templates carry the dojo tooling version. A version change invalidates old receipts and prevents an outdated template from launching. `dojo_version()` reports the installed version:

In [ ]:
from llmdojo.dojo import dojo_version

In [ ]:
dojo_version()

'0.0.1:3'

### Limitations

A tooling rename does not change the dojo version automatically. Rebuild templates whenever names used by the worked round or its documentation change.

Doc-state host detection has two conservative fallbacks. Two simultaneous Claude conversations in the same project can select the wrong transcript record. Codex's MCP worker is keyed by parent process id, so compaction clears all numeric records; another live Codex session may receive extra documentation reminders. The reminders are harmless.